In [ ]:
import qutip as q
import matplotlib.pyplot as plt
import numpy as np
%matplotlib inline

In [ ]:
def hamiltonian(Ec, Ej, N, phi, ng, Ej2 = 0):
    """
    Return the charge qubit hamiltonian as a Qobj instance.
    """
    # n = (Ej/(32*Ec))**0.25
   # phi = (2*Ec/Ej)**0.25
    gamma = Ej2 / Ej
    d = (gamma - 1) / (gamma + 1)

    # m = np.sqrt(8*Ec*Ej)*(np.diag(np.sqrt(np.arange(2*N)), -1)*np.diag(np.sqrt(np.arange(2*N)), 1)+0.5)
    # m = np.diag(4 * Ec * (np.arange(-N,N+1)-ng)**2) + 0.5 * Ej * (np.diag(-np.ones(2*N), 1) + 
    #                                                            np.diag(-np.ones(2*N), -1))

    m = Ec * (q.destroy(N) * q.create(N) + 0.5)
    
    return q.Qobj(m)

def lc_oscillator(wq, N):
    H = wq * (q.create(N) * q.destroy(N)  + 0.5)

    return q.Qobj(H)

def transmon1(wq, Ec, N):
    H = wq * ( q.create(N) * q.destroy(N)) - (Ec / 2) * (q.create(N) * q.create(N) * q.destroy(N) * q.destroy(N))

    return q.Qobj(H)

def transmon2(Ec, Ej, ng, N):
    H = np.diag(4 * Ec * (np.arange(-N,N+1)-ng)**2) + 0.5 * Ej * (np.diag(-np.ones(2*N), 1) + np.diag(-np.ones(2*N), -1))

    return q.Qobj(H)

def split_transmon(Ec, Ej, ng, phi, N):
    H = np.diag(4 * Ec * (np.arange(-N,N+1)-ng)**2) + Ej * np.abs(np.cos(phi)) * (np.diag(-np.ones(2*N), 1) + np.diag(-np.ones(2*N), -1))

    return q.Qobj(H)

def assymetric_transmon(Ec, Ej1, Ej2, ng, phi, N):
    gamma = Ej2 / Ej1
    d = (gamma - 1) / (gamma + 1)
    H = np.diag(4 * Ec * (np.arange(-N,N+1)-ng)**2) + (Ej1 + Ej2) * np.sqrt(np.cos(phi)**2 + (d**2)*np.sin(phi)**2) * (np.diag(-np.ones(2*N), 1) + np.diag(-np.ones(2*N), -1))

    return q.Qobj(H)

def jaynes_cummings(wr, wq, g, N): 
    a  = q.tensor(q.destroy(N), q.qeye(2))
    sm = q.tensor(q.qeye(N), q.destroy(2))

    H = wr * a.dag() * a + 0.5 * wq * q.tensor(q.qeye(N), q.sigmaz()) + g * (a.dag() * sm + a * sm.dag())
    
    return q.Qobj(H)
     

In [ ]:
def plot_energies(ng_vec, energies, ymax=(20, 3)):
    """
    Plot energy levels as a function of bias parameter ng_vec.
    """
    fig, axes = plt.subplots(1,2, figsize=(16,6))

    for n in range(len(energies[0,:])):
        axes[0].plot(ng_vec, energies[:,n])
    axes[0].set_ylim(-20, ymax[0])
    axes[0].set_xlabel(r'n', fontsize=18)
    axes[0].set_ylabel(r'E', fontsize=18)

    # for n in range(len(energies[0,:])):
    #     axes[1].plot(ng_vec, (energies[:,n]-energies[:,0])/(energies[:,1]-energies[:,0]))
    axes[1].plot(ng_vec, (energies[:,1]-energies[:,0]))
    axes[1].plot(ng_vec, (energies[:,2]-energies[:,1]))

    axes[1].set_ylim(-0.1, ymax[1])
    axes[1].set_xlabel(r'n', fontsize=18)
    axes[1].set_ylabel(r'E10, E21', fontsize=18)
    return fig, axes

In [ ]:
    
N = 10  
Ec = 0.3  
Ej = 15
        
    
ng_vec = np.linspace(-4, 4, 200)

energies = (np.array([transmon2(Ec, Ej, 0, N).eigenenergies() for ng in ng_vec]))

# energies = (np.array([assymetric_transmon(Ec, Ej, 2*Ej, 0, ng, N).eigenenergies() for ng in ng_vec]))

plot_energies(ng_vec, energies, ymax=(20, 30))